<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/BaselineTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --force-reinstall numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 72.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 re

In [1]:
import os
import numpy as np

from google.colab import drive
drive.mount("/content/drive", force_remount = False)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

import random

from torch.utils.data import Dataset

#import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
#CUDA_version = torch.version.cuda.replace('.', '')

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

#import torch_sparse

from scipy.ndimage import maximum_filter

import sys

#!pip install import-ipynb
#import import_ipynb

#!pip install open3d plotly
#import open3d as o3d

import matplotlib.pyplot as plt


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!npx degit google-research-datasets/Objectron/objectron objectron

sys.path.insert(0, '/content')

from objectron.dataset.iou import IoU as IoU3d
from objectron.dataset.box import Box as BoxIoU

⠙⠹⠸⠼⠴⠦Need to install the following packages:
degit@3.8.0
Ok to proceed? (y) y

⠙⠹> cloned google-research-datasets/Objectron#HEAD to objectron
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠙

In [3]:
!pip install --force-reinstall opencv-python-headless==4.9.0.80 &> /dev/null
!pip install nuscenes-devkit &> /dev/null

from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud, Box
from nuscenes.eval.detection.utils import category_to_detection_name
from nuscenes.utils.geometry_utils import points_in_box

nusc_root = "/content/drive/MyDrive/LiDARFusion/nuscenes/datanuscenes"

nusc = NuScenes(version='v1.0-mini', dataroot=nusc_root, verbose=True)

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 6.214 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


In [4]:
from pyquaternion import Quaternion

sys.path.insert(0, '/content/drive/MyDrive/LiDARFusion')

sys.path.append('/content/objectron/')

from voxel_pointnet2 import PointNetSetAbstraction

from centerpoint import CenterPoint

from lidar_baseline import PillarsEncoder, get_ij, get_point_pillar

import lidartrainlibrary as ltb

#import CenterPointHead

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
class LIDARBaseline(nn.Module):

  def __init__(self, encoder, extracter):
    super().__init__()

    self.encoder = encoder
    self.extracter = extracter

  def forward(self, pc):
    out = self.encoder(pc)

    boxes = self.extracter(out)

    return boxes

In [6]:
Encoder = PillarsEncoder(extra_features=2)
BoxHead = CenterPoint(512,512,256,10, K = 50)
model = LIDARBaseline(Encoder, BoxHead)

In [7]:
import pickle

dict_file_path = '/content/drive/MyDrive/LiDARFusion/gt_dict.pkl'

with open(dict_file_path, 'rb') as f:
    gt_paste = pickle.load(f)


In [27]:
class_names = ["barrier", "bicycle", "bus", "car", "construction_vehicle", "motorcycle", "pedestrian", "traffic_cone", "trailer", "truck"]

def check_bev_iou(new_box, existing):
  new_box_corners = new_box.corners()[0:2, [0,1,4,5]]

  for box in existing:
    corners = box.corners()[0:2,[0,1,4,5]]

    corners_x , corners_y = corners

    for coords in new_box_corners.T:
      x,y = coords

      if corners_x[corners_x<x].size > 0 and corners_x[corners_x>x].size > 0:
        if corners_y[corners_y<y].size >0 and corners_y[corners_y>y].size >0:

          return True

  return False

def sample_valid_placement(gt_boxes):
  x_min, y_min, x_max, y_max, z_min, z_max = ltb.get_boxes_max_min(gt_boxes)

  x_cand = np.random.uniform(x_min, x_max)

  y_cand = np.random.uniform(y_min, y_max)

  z_cand = np.random.uniform(z_min, z_max)

  return np.array([x_cand, y_cand, z_cand])

def gt_sampling(cloud, gt_boxes, gt_names, gt_dict, sample_groups): #sample_groups is dict of how many of each class to sample

  for cls, num_sample in sample_groups.items():
    candidates = random.sample(gt_dict[cls], num_sample)

    for cand in candidates:
      new_center = sample_valid_placement(gt_boxes)
      new_box = cand["box"]
      new_box.translate(new_center)

      detection_name = ltb.category_to_detection_name(new_box.name)
      new_box.name = detection_name

      label = ltb.category_to_label(new_box.name)
      new_box.label = label

      if check_bev_iou(new_box, gt_boxes):
        continue

      new_points = cand['points'].copy()
      new_points[:, :3] += new_center

      points = np.concatenate([cloud, new_points], axis=0)
      gt_boxes.append(new_box)
      gt_names.append(cls)

  return points, gt_boxes, gt_names

In [9]:
def get_box_names(boxes):

  labels = []

  for i in range(len(boxes)):

    name = boxes[i].name

    labels.append(name)

  return labels


def get_samples(scenes, nusc):

  samples = []

  for scene in scenes:
    scene_0 = nusc.scene[scene]
    token_0 = scene_0["first_sample_token"]

    while token_0 != "":

      token_sample = nusc.get("sample", token_0)
      samples.append(token_sample)
      token_0 = token_sample["next"]

  return samples


In [10]:
test_scene = nusc.scene[0]

token_0 = test_scene["first_sample_token"]

cloud_sample = nusc.get("sample", token_0)

boxes = nusc.get_boxes(cloud_sample["data"]["LIDAR_TOP"])

points = ltb.load_sweeps(nusc, cloud_sample)

gt_boxes = ltb.process_boxes(nusc, cloud_sample, boxes, points)

gt_names = get_box_names(gt_boxes)

sample_groups = {"car": 2, "pedestrian": 1, "construction_vehicle": 4}

samples = get_samples([0,1,2], nusc)

In [26]:
random.sample(gt_paste["car"], 1)[0]["box"]

label: nan, score: nan, xyz: [10.34, 14.57, 0.40], wlh: [2.56, 5.64, 2.23], rot axis: [-0.40, 0.18, 0.90], ang(degrees): -11.84, ang(rad): -0.21, vel: nan, nan, nan, name: vehicle.car, token: 78b0b6f0a3db4536b4f24da9c17c2523

In [53]:
class LidarDetectionDataset(Dataset): #contains only the nusc samples from which points/boxes are pulled

  def __init__(self, samples, gt_paste_samplegroups):

    self.samples = samples

    self.sample_groups = gt_paste_samplegroups

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):

    sample_idx = self.samples[idx]

    points = ltb.load_sweeps(nusc, sample_idx)
    boxes = nusc.get_boxes(sample_idx["data"]["LIDAR_TOP"])
    boxes = ltb.process_boxes(nusc, sample_idx, boxes, points)
    names = get_box_names(boxes)

    points, boxes, names = gt_sampling(points, boxes, names, gt_paste, self.sample_groups)

    ltb.global_augmentation(points, boxes)

    pillars, pillars_mask = ltb.get_pillars(points)

    points = torch.from_numpy(points)

    heatmap, offset, size, z, rotation, index, mask, cat = ltb.target_generation(boxes)

    reg_targets = [offset, size, z, rotation]

    return {
        "pillars": pillars,
        "pilars_mask": pillars_mask,
        "gt_boxes": boxes,
        "cloud": points,
        "gt_labels": names,
        "heatmap": heatmap,
        "reg_target": reg_targets,
        "ind": index,
        "targets_mask": mask,
        "cat": cat
    }

In [54]:
lidarset = LidarDetectionDataset(samples, sample_groups)